In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('FakeNewsNet.csv',engine='python')

In [ ]:
df.head()

In [5]:
df.isnull().sum()

,0
title,0
news_url,330
source_domain,330
tweet_num,0
real,0


In [ ]:
df.dropna()

In [7]:
X=df.drop('source_domain',axis=1)

In [8]:
y=df['source_domain']

In [9]:
y

,source_domain
0,toofab.com
1,www.today.com
2,www.etonline.com
3,www.dailymail.co.uk
4,www.zerchoo.com
...,...
23191,www.express.co.uk
23192,hollywoodlife.com
23193,www.justjared.com
23194,www.intouchweekly.com


In [10]:
import tensorflow as tf

In [11]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [12]:
voc_size=5000

In [13]:
message = X.copy()

In [14]:
message['title'][1]

"People's Choice Awards 2018: The best red carpet looks"

In [15]:
import nltk
import re
from nltk.corpus import stopwords

In [16]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [17]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0,len(message)):
  review = re.sub('[^a-zA-Z]',' ',message['title'][i])
  review = review.lower()
  review = review.split()

  review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)
  corpus.append(review)

In [18]:
onehot_rep = [one_hot(words,voc_size) for words in corpus]

In [ ]:
onehot_rep

In [20]:
sent=20
embed = pad_sequences(onehot_rep,padding='pre',maxlen=sent)

In [22]:
embed[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0, 1157,
       3557, 4565, 2393, 3867, 3308, 3179, 3920,  318, 1742], dtype=int32)

In [23]:
embed_feature = 40
model = Sequential()
model.add(Embedding(voc_size,embed_feature,input_length=sent))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)